# 03 — ML training and reconstruction dashboard

This notebook is the heart of CALOMAPS:

1. **Train** (or load) a Deep Quantile Ensemble surrogate model for each of the 4 readouts (Analog, MIP, Hits, Cluster).
2. **Reconstruct** energy via Neyman inversion of the surrogate.
3. **Plot** the 3-panel physics dashboard.

You need a GPU for training. See [`docs/handbook.md`](../docs/handbook.md) §11.2 for the CUDA torch setup recipe. If you don't want to retrain, you can load the shipped ensembles in `models/saved_ensembles_gpu_v2/` directly (skip the training cell).

**Kernel**: `Key4hep + GPU`. Verify GPU availability in the very first cell.


## 1. Verify the kernel + GPU

In [ ]:
import torch, sys
print(f"torch version: {torch.__version__}")
print(f"torch path:    {torch.__file__}")
print(f"cuda built:    {torch.backends.cuda.is_built()}")
print(f"cuda available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print()
    print("WARNING: no CUDA. You're on a CPU-only torch — training will take ~30 min instead of ~9 min.")
    print("To enable GPU: run the following in a cell (then RESTART KERNEL):")
    print("  !pip install --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")


## 2. Load the extracted dataset

In [ ]:
import os, numpy as np
CALOMAPS_HOME = os.environ.get("CALOMAPS_HOME", os.path.expanduser("~/CALOMAPS"))

# Make our analysis utilities importable
sys.path.insert(0, os.path.join(CALOMAPS_HOME, "analysis"))
from quantilenet import QuantileNet, quantile_loss, save_ensemble, load_ensemble, QUANTILES
from dashboard import get_interpolators, reco_metrics_over_grid, plot_dashboard

npz_path = os.path.join(CALOMAPS_HOME, "models", "decal_extracted_data.npz")
data = np.load(npz_path)
all_truth   = data["all_truth"]
all_visible = data["all_visible"]
all_mip     = data["all_mip"]
all_hits    = data["all_hits"]
all_cluster = data["all_cluster"]
print(f"loaded {len(all_truth)} events from {npz_path}")

valid = (all_hits > 0) & (all_truth > 0) & (all_visible > 0) & (all_mip > 0) & (all_cluster > 0)
x_train = all_truth[valid]
print(f"valid events for training: {valid.sum()}")


## 3. Either load pre-trained ensembles **or** train fresh ones

Set `RETRAIN = False` to skip the (~10 min) training step and use shipped checkpoints.
Set `RETRAIN = True` to retrain — output goes to `$CALOMAPS_HOME/models/saved_ensembles_gpu_v2/`.


In [ ]:
from quantilenet import train_one_ensemble

RETRAIN = False   # set True to retrain from scratch
ENSEMBLE_DIR = os.path.join(CALOMAPS_HOME, "models", "saved_ensembles_gpu_v2")
os.makedirs(ENSEMBLE_DIR, exist_ok=True)

if RETRAIN:
    # Train all 4 ensembles inline. Takes ~10 min on the A100, ~30-60 min on CPU.
    # Hyperparameters (num_models, epochs, lr, patience) can be passed to
    # train_one_ensemble — see analysis/quantilenet.py for defaults.
    for label, y_arr, seed, fname in [
        ("True Analog",      all_visible, 1000, "ens_analog.pth"),
        ("MIP Proxy",        all_mip,     2000, "ens_mip.pth"),
        ("Raw Hits",         all_hits,    3000, "ens_hits.pth"),
        ("Naive Clustering", all_cluster, 4000, "ens_cluster.pth"),
    ]:
        ens, xmax, ymax = train_one_ensemble(x_train, y_arr[valid], device,
                                              name=label, seed_base=seed)
        save_ensemble(ens, xmax, ymax, os.path.join(ENSEMBLE_DIR, fname))
else:
    print(f"Skipping training. Loading existing ensembles from {ENSEMBLE_DIR}")

# Load all 4 ensembles (either freshly trained above or pre-shipped)
ens_a, xa, ya = load_ensemble(os.path.join(ENSEMBLE_DIR, "ens_analog.pth"), device)
ens_m, xm, ym = load_ensemble(os.path.join(ENSEMBLE_DIR, "ens_mip.pth"),     device)
ens_h, xh, yh = load_ensemble(os.path.join(ENSEMBLE_DIR, "ens_hits.pth"),    device)
ens_c, xc, yc = load_ensemble(os.path.join(ENSEMBLE_DIR, "ens_cluster.pth"), device)
print(f"loaded 4 ensembles, {len(ens_a)} models each, on device={device}")


## 4. Build forward interpolators and run Neyman reconstruction

In [ ]:
fl_a, fm_a, fh_a = get_interpolators(ens_a, xa, ya, device)
fl_m, fm_m, fh_m = get_interpolators(ens_m, xm, ym, device)
fl_h, fm_h, fh_h = get_interpolators(ens_h, xh, yh, device)
fl_c, fm_c, fh_c = get_interpolators(ens_c, xc, yc, device)

reco = {
    "Analog":  reco_metrics_over_grid(fl_a, fm_a, fh_a),
    "MIP":     reco_metrics_over_grid(fl_m, fm_m, fh_m),
    "Hits":    reco_metrics_over_grid(fl_h, fm_h, fh_h),
    "Cluster": reco_metrics_over_grid(fl_c, fm_c, fh_c),
}

# Print headline numbers
et = reco["Analog"][0]
print("\n=== reconstructed resolutions (sigma_reco / E_true) ===")
for energy in (10, 100, 300):
    idx = np.argmin(np.abs(et - energy))
    print(f"  E={energy:>3d} GeV:  ", end="")
    for key in ("Analog", "MIP", "Hits", "Cluster"):
        print(f"{key.lower()}={reco[key][2][idx]:.4f}  ", end="")
    print()


## 5. The 3-panel physics dashboard

In [ ]:
plot_dashboard(reco, out_path_prefix=None, show=True)


## What to look for in the dashboard

**Panel 1 — Linearity** (`E_reco / E_true`): should sit flat at 1.0. *This is partially trivial* (we're inverting the median surrogate with itself) — see [`docs/handbook.md`](../docs/handbook.md) §13.1 for caveats.

**Panel 2 — Resolution** (`σ_reco / E_true` vs `E_true`): the DECAL physics. Analog and MIP curves should overlap (they're the "ideal" readouts). Raw Hits should diverge upward at high energy as **pixel saturation** kicks in — that's the saturation breakdown around ~100 GeV at 100 µm pixel pitch.

**Panel 3 — Stochastic** (`σ_reco / E_true` vs `1/√E_true`): straight line through origin = pure stochastic resolution `a/√E`. Deviations on the right side (low E) = sampling-fluctuation regime. Curving-up on the left (high E) = constant-term contributions from saturation/leakage.

For physics interpretation see [`docs/DECAL_pipeline.md`](../docs/DECAL_pipeline.md) §6 and [`docs/handbook.md`](../docs/handbook.md) §13.


## Suggested follow-up experiments

1. **Pixel pitch scan**: re-run the entire pipeline at 25, 50, 200 µm pixel pitch (edit `ECal_cell_size` in `geometry/SiD_TestBeam.xml`, regenerate sim, re-extract, retrain). Plot how the saturation knee moves.
2. **Particle type**: change `gun.particle` in `sim/run_sim.py` to `"pi+"` or `"proton"`. Hadronic showers spread differently — how does the resolution change?
3. **More models per ensemble**: edit `num_models=20` in `analysis/train_ensembles.py`. Does the resolution prediction tighten with more bootstrap samples?
4. **Held-out test set**: the current linearity panel is trivially perfect because we use the same curve for forward and inverse. Modify the pipeline to bootstrap-resample real events and reconstruct each one — what does linearity look like then?
